## Gmail ingest with attachments

Forked from Langchains Ambient Agents project. 
Purpose is to ingest gmail emails (specifying necessary filters on which emails should be ingested)
But adds in grabbing any attachements and storing them in a GCS bucket

Emails are then added to a langgraph graph based on a thread-id derived from (but not identical to) the thread_id in the Gmail search.

Right now, we are only accepting PDF attachments.

**If you see `invalid_grant: Bad Request`:** The refresh token in `.secrets/token.json` is expired or revoked (e.g. OAuth app in Testing mode, or access revoked). Re-run the OAuth flow: `uv run python src/gcp/setup_gmail.py` from project root (requires `.secrets/secrets.json`). A browser will open to re-authorize and a new token will be saved.

TODOS :

Abstract away the Google Authentication into a separate service.
Note we need to separate services 1 for Gmail (own set of secrets) and 1 for GCP.

If you see `invalid_grant: Bad Request`

This means the **refresh token** in `.secrets/token.json` is no longer valid. e.g.

- **OAuth app in "Testing"** – Google expires refresh tokens after 7 days of user inactivity. Re-authorize to get a new token.
- **Access revoked** – User removed the app in [Google Account → Security → Third-party access](https://myaccount.google.com/connections).
- **Client ID/secret changed** – Token was issued with different OAuth credentials.

**Fix:** Re-run the Gmail OAuth flow so a new token is saved to `.secrets/token.json`:

```bash
# From project root; requires .secrets/secrets.json (OAuth client JSON from Google Cloud Console)
uv run python src/gcp/setup_gmail.py
```

A browser will open to sign in and approve access; the new token is then written to `.secrets/token.json`.

In [ ]:

import base64
import json
import uuid
import hashlib
import asyncio
import argparse
import os
from pathlib import Path
from datetime import datetime
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from google.cloud import storage
from langgraph_sdk import get_client



# Setup paths - Source File approach
# _ROOT = Path(__file__).parent.absolute()
# _SECRETS_DIR = _ROOT / ".secrets"
#TOKEN_PATH = _SECRETS_DIR / "token.json"

# Notebook
_SECRETS_DIR=Path.cwd().parent / ".secrets"
TOKEN_PATH = _SECRETS_DIR / "token.json"

# GCP
BUCKET_NAME="fionaa-customer-assets"
PROJECT_ID=;

In [12]:
async def ingest_email_to_langgraph(email_data, graph_name, url="http://127.0.0.1:2024"):
    """Ingest an email to LangGraph."""
    # Connect to LangGraph server
    client = get_client(url=url)
    
    # Create a consistent UUID for the thread
    raw_thread_id = email_data["thread_id"]
    thread_id = str(
        uuid.UUID(hex=hashlib.md5(raw_thread_id.encode("UTF-8")).hexdigest())
    )
    print(f"Gmail thread ID: {raw_thread_id} → LangGraph thread ID: {thread_id}")
    
    thread_exists = False
    try:
        # Try to get existing thread info
        thread_info = await client.threads.get(thread_id)
        thread_exists = True
        print(f"Found existing thread: {thread_id}")
    except Exception as e:
        # If thread doesn't exist, create it
        print(f"Creating new thread: {thread_id}")
        thread_info = await client.threads.create(thread_id=thread_id)
    
    # If thread exists, clean up previous runs
    if thread_exists:
        try:
            # List all runs for this thread
            runs = await client.runs.list(thread_id)
            
            # Delete all previous runs to avoid state accumulation
            for run_info in runs:
                run_id = run_info.id
                print(f"Deleting previous run {run_id} from thread {thread_id}")
                try:
                    await client.runs.delete(thread_id, run_id)
                except Exception as e:
                    print(f"Failed to delete run {run_id}: {str(e)}")
        except Exception as e:
            print(f"Error listing/deleting runs: {str(e)}")
    
    # Update thread metadata with current email ID
    await client.threads.update(thread_id, metadata={"email_id": email_data["id"]})
    
    # Create a fresh run for this email
    print(f"Creating run for thread {thread_id} with graph {graph_name}")
    
    # Build email input with PDF attachment paths if available
    email_input = {
        "from": email_data["from_email"],
        "to": email_data["to_email"],
        "subject": email_data["subject"],
        "body": email_data["page_content"],
        "id": email_data["id"]
    }
    
    # Add PDF attachment paths if they exist (schema now supports this field)
    if "pdf_attachments" in email_data and email_data["pdf_attachments"]:
        email_input["pdf_attachments"] = email_data["pdf_attachments"]
        print(f"  Including {len(email_data['pdf_attachments'])} PDF attachment(s) in LangGraph input")
        print(f"  PDF attachment paths: {email_data['pdf_attachments']}")
    
    try:
        run = await client.runs.create(
            thread_id,
            graph_name,
            input={"email_input": email_input},
            multitask_strategy="rollback",
        )
    except Exception as e:
        # Print more detailed error information
        import json
        print(f"\n  ⚠️  Error creating run:")
        print(f"  Input keys: {list(email_input.keys())}")
        print(f"  PDF attachments included: {'pdf_attachments' in email_input}")
        print(f"  Error type: {type(e).__name__}")
        print(f"  Error message: {str(e)}")
        
        # Try to get more details from the exception
        if hasattr(e, 'response'):
            try:
                if hasattr(e.response, 'text'):
                    print(f"  Response text: {e.response.text}")
                if hasattr(e.response, 'json'):
                    print(f"  Response JSON: {json.dumps(e.response.json(), indent=2)}")
            except:
                pass
        
        # Print the actual input structure (without the body content which might be long)
        input_preview = {k: (v if k != 'body' else f"[{len(str(v))} chars]") for k, v in email_input.items()}
        print(f"  Input preview: {json.dumps(input_preview, indent=2)}")
        print()
        raise
    
    print(f"Run created successfully with thread ID: {thread_id}")
    
    return thread_id, run

In [13]:
# GCP Authentication Setup for Google Cloud Storage
# Run this cell to set up authentication for GCS

def setup_gcs_authentication():
    """
    Set up GCP authentication for Google Cloud Storage.
    
    This function tries multiple authentication methods:
    1. Service account key file in .secrets/gcs-service-account.json
    2. GOOGLE_APPLICATION_CREDENTIALS environment variable
    3. Application Default Credentials (requires gcloud CLI)
    """

    # WRONG PATH
    service_account_path = Path.cwd().parent / "src" / "email_assistant" / "tools" / "gmail" / ".secrets" / "gcs-service-account.json"
    
    # Method 1: Check for service account key file
    if service_account_path.exists():
        os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(service_account_path)
        print(f"✓ Found service account key at: {service_account_path}")
        return True
    
    # Method 2: Check if GOOGLE_APPLICATION_CREDENTIALS is already set
    if os.getenv("GOOGLE_APPLICATION_CREDENTIALS"):
        print(f"✓ Using GOOGLE_APPLICATION_CREDENTIALS: {os.getenv('GOOGLE_APPLICATION_CREDENTIALS')}")
        return True
    
    # Method 3: Try to use Application Default Credentials
    try:
        from google.auth import default
        credentials, project = default()
        print(f"✓ Using Application Default Credentials (project: {project})")
        return True
    except Exception as e:
        print(f"✗ No authentication found. Error: {str(e)}")
      
        return False

# Run authentication setup
setup_gcs_authentication()


✓ Using GOOGLE_APPLICATION_CREDENTIALS: ./.secrets/fionaa-service-acct.json


True

In [14]:
def load_gmail_credentials():
    """
    Load Gmail credentials from token.json or environment variables.
    
    This function attempts to load credentials from multiple sources in this order:
    1. Environment variables GMAIL_TOKEN
    2. Local file at token_path (.secrets/token.json)
    
    Returns:
        Google OAuth2 Credentials object or None if credentials can't be loaded
    """
    token_data = None
    
    # 1. Try environment variable
    env_token = os.getenv("GMAIL_TOKEN")
    if env_token:
        try:
            token_data = json.loads(env_token)
            print("Using GMAIL_TOKEN environment variable")
        except Exception as e:
            print(f"Could not parse GMAIL_TOKEN environment variable: {str(e)}")
    
    # 2. Try local file as fallback
    if token_data is None:
        if TOKEN_PATH.exists():
            try:
                with open(TOKEN_PATH, "r") as f:
                    token_data = json.load(f)
                print(f"Using token from {TOKEN_PATH}")
            except Exception as e:
                print(f"Could not load token from {TOKEN_PATH}: {str(e)}")
        else:
            print(f"Token file not found at {TOKEN_PATH}")
    
    # If we couldn't get token data from any source, return None
    if token_data is None:
        print("Could not find valid token data in any location")
        return None
    
    try:
        # Create credentials object
        credentials = Credentials(
            token=token_data.get("token"),
            refresh_token=token_data.get("refresh_token"),
            token_uri=token_data.get("token_uri", "https://oauth2.googleapis.com/token"),
            client_id=token_data.get("client_id"),
            client_secret=token_data.get("client_secret"),
            scopes=token_data.get("scopes", ["https://www.googleapis.com/auth/gmail.modify"])
        )
        return credentials
    except Exception as e:
        print(f"Error creating credentials object: {str(e)}")
        return None

In [15]:
def extract_message_part(payload):
    """Extract content from a message part."""
    # If this is multipart, process with preference for text/plain
    if payload.get("parts"):
        # First try to find text/plain part
        for part in payload["parts"]:
            mime_type = part.get("mimeType", "")
            if mime_type == "text/plain" and part.get("body", {}).get("data"):
                data = part["body"]["data"]
                return base64.urlsafe_b64decode(data).decode("utf-8")
                
        # If no text/plain found, try text/html
        for part in payload["parts"]:
            mime_type = part.get("mimeType", "")
            if mime_type == "text/html" and part.get("body", {}).get("data"):
                data = part["body"]["data"]
                return base64.urlsafe_b64decode(data).decode("utf-8")
                
        # If we still haven't found content, recursively check for nested parts
        for part in payload["parts"]:
            content = extract_message_part(part)
            if content:
                return content
    
    # Not multipart, try to get content directly
    if payload.get("body", {}).get("data"):
        data = payload["body"]["data"]
        return base64.urlsafe_b64decode(data).decode("utf-8")

    return ""


In [24]:
def extract_pdf_attachments(service, message_id, payload, thread_id=None, bucket_name=BUCKET_NAME, project_id=PROJECT_ID):
    """
    Extract PDF attachments from a Gmail message and upload them to Google Cloud Storage.
    
    Args:
        service: Gmail API service object
        message_id: The Gmail message ID
        payload: The message payload object
        thread_id: Optional thread ID to organize files in GCS. If provided, PDFs will be saved to https://storage.googleapis.com/bucket_name/thread_id/
        bucket_name: GCS bucket name (default: "aim-fiona")
        project_id: GCP project ID (default: "gcp-tutorials-218608")
    
    Returns:
        List of dictionaries containing attachment info: [{'filename': str, 'filepath': str (GCS HTTPS URL), 'size': int}]
    """
    # Initialize GCS client
    # storage_client = storage.Client(project=project_id)
    # bucket = storage_client.bucket(bucket_name)
    
    # # Create blob path prefix with thread_id if provided
    # if thread_id:
    #     blob_prefix = f"{thread_id}/"
    # else:
    #     blob_prefix = ""
    
    # print(f"  Uploading PDFs to GCS: https://storage.googleapis.com/{bucket_name}/{blob_prefix}")
    
    pdf_attachments = []
    
    def find_pdf_parts(part, parts_list=None):
        """Recursively find all PDF parts in the message."""
        if parts_list is None:
            parts_list = []
        
        mime_type = part.get("mimeType", "")
        
        # Check if this part is a PDF attachment
        if mime_type == "application/pdf":
            attachment_id = part.get("body", {}).get("attachmentId")
            filename = part.get("filename", "unnamed.pdf")
            
            if attachment_id:
                parts_list.append({
                    "attachment_id": attachment_id,
                    "filename": filename,
                    "size": part.get("body", {}).get("size", 0)
                })
                print(f"  Found PDF attachment: {filename} (attachmentId: {attachment_id})")
            else:
                print(f"  Warning: PDF part found but no attachmentId for {filename}")
        
        # Recursively check nested parts
        if part.get("parts"):
            for sub_part in part["parts"]:
                find_pdf_parts(sub_part, parts_list)
        
        return parts_list
    
    # Find all PDF parts
    pdf_parts = find_pdf_parts(payload)
    
    if not pdf_parts:
        return pdf_attachments
    
    # Download each PDF attachment
    for pdf_part in pdf_parts:
        try:
            attachment_id = pdf_part["attachment_id"]
            filename = pdf_part["filename"]
            size = pdf_part["size"]
            
            print(f"  Downloading attachment: {filename} (ID: {attachment_id})")
            
            # Get the attachment data
            attachment = service.users().messages().attachments().get(
                userId="me",
                messageId=message_id,
                id=attachment_id
            ).execute()
            
            # Decode the attachment data (it's base64url encoded)
            attachment_data = base64.urlsafe_b64decode(attachment["data"])
            
            # Verify we got data
            if not attachment_data:
                print(f"  Warning: No data received for {filename}")
                continue
            
            print(f"  Decoded {len(attachment_data)} bytes of attachment data")
            
            # Write the PDF attachment data to a local file; ensure to open in binary mode
            # Create the output directory if it doesn't exist
            output_dir = '../data/test/'
            import os
            os.makedirs(output_dir, exist_ok=True)
            output_path = os.path.join(output_dir, filename)
            with open(output_path, 'wb') as file:
                file.write(attachment_data)


            # TODO -GCloud
            # Create a safe blob name (handle duplicate names by appending counter)
            # safe_filename = filename
            # counter = 1
            # blob_name = f"{blob_prefix}{safe_filename}"
            
            # # Check if blob already exists and append counter if needed
            # while bucket.blob(blob_name).exists():
            #     name_part = Path(filename).stem
            #     ext_part = Path(filename).suffix
            #     safe_filename = f"{name_part}_{counter}{ext_part}"
            #     blob_name = f"{blob_prefix}{safe_filename}"
            #     counter += 1
            
            # # Create blob and upload the PDF file
            # blob = bucket.blob(blob_name)
            # blob.upload_from_string(attachment_data, content_type='application/pdf')
            
            # # Get the GCS URL (HTTPS URL)
            # gcs_url = f"https://storage.googleapis.com/{bucket_name}/{blob_name}"
            
            # pdf_attachments.append({
            #     "filename": filename,
            #     "filepath": gcs_url,
            #     "size": len(attachment_data)
            # })
            # print(f"  ✓ Uploaded PDF: {filename} ({len(attachment_data)} bytes) -> {gcs_url}")
            
        except Exception as e:
            import traceback
            print(f"  ✗ Error extracting PDF attachment {pdf_part.get('filename', 'unknown')}: {str(e)}")
            print(f"    Traceback: {traceback.format_exc()}")
    
    return pdf_attachments


In [25]:
def extract_email_data(message):
    """Extract key information from a Gmail message."""
    headers = message['payload']['headers']
    
    # Extract key headers
    subject = next((h['value'] for h in headers if h['name'] == 'Subject'), 'No Subject')
    from_email = next((h['value'] for h in headers if h['name'] == 'From'), 'Unknown Sender')
    to_email = next((h['value'] for h in headers if h['name'] == 'To'), 'Unknown Recipient')
    date = next((h['value'] for h in headers if h['name'] == 'Date'), 'Unknown Date')
    
    # Extract message content
    content = extract_message_part(message['payload'])
    
    # Create email data object
    email_data = {
        "from_email": from_email,
        "to_email": to_email,
        "subject": subject,
        "page_content": content,
        "id": message['id'],
        "thread_id": message['threadId'],
        "send_time": date
    }
    
    return email_data

In [ ]:
async def fetch_and_process_emails(args):
    attachments = []
    """Fetch emails from Gmail and process them through LangGraph."""
    # Load Gmail credentials
    credentials = load_gmail_credentials()
    if not credentials:
        print("Failed to load Gmail credentials")
        return 1
        
    # Build Gmail service
    service = build("gmail", "v1", credentials=credentials)
    
    # Process emails
    processed_count = 0
    
    try:
        # Get messages from the specified email address
        email_address = args.email
        
        # Construct Gmail search query
        query = f"to:{email_address} OR from:{email_address}"
        
        # Add time constraint if specified
        if args.minutes_since > 0:
    
            # Calculate timestamp for filtering
            from datetime import timedelta
            after = int((datetime.now() - timedelta(minutes=args.minutes_since)).timestamp())
            query += f" after:{after}"
            
        # Only include unread emails unless include_read is True
        if not args.include_read:
            query += " is:unread"
            
        print(f"Gmail search query: {query}")
        
        # Execute the search
        results = service.users().messages().list(userId="me", q=query).execute()
        messages = results.get("messages", [])
        
        if not messages:
            print("No emails found matching the criteria")
            return 0
            
        print(f"Found {len(messages)} emails")
        
        # Process each email
        for i, message_info in enumerate(messages):
            # Stop early if requested
            if args.early and i > 0:
                print(f"Early stop after processing {i} emails")
                break
                
                
            # Get the full message with format='full' to ensure attachment metadata is included
            message = service.users().messages().get(
                userId="me", 
                id=message_info["id"],
                format='full'
            ).execute()
            
            # Extract email data
            email_data = extract_email_data(message)
            
            print(f"\nProcessing email {i+1}/{len(messages)}:")
            print(f"From: {email_data['from_email']}")
            print(f"Subject: {email_data['subject']}")
            
            # Extract PDF attachments - save to subdirectory named with thread_id
            pdf_attachments = extract_pdf_attachments(
                service, 
                message_info["id"], 
                message['payload'],
                thread_id=email_data['thread_id']
            )
            
            # Add PDF attachment paths to email_data for LangGraph
            if pdf_attachments:
                # Extract just the filepaths for LangGraph
                email_data['pdf_attachments'] = [att['filepath'] for att in pdf_attachments]
                print(f"  Found {len(pdf_attachments)} PDF attachment(s)")
                attachments.extend(pdf_attachments)
            else:
                email_data['pdf_attachments'] = []
                print(f"  No PDF attachments found")
            
            # # TOD: Ingest to LangGraph
            # thread_id, run = await ingest_email_to_langgraph(
            #     email_data, 
            #     args.graph_name,
            #     url=args.url
            # )
            # processed_count += 1
            
        print(f"\nProcessed {processed_count} emails successfully")
        return 0
        
    except Exception as e:
        print(f"Error processing emails: {str(e)}")
        return 1


In [1]:


# class Args:
#     def __init__(self):
#         self.email = "stevejgoodman@gmail.com"
#         self.minutes_since = 10
#         self.include_read = True
#         self.early = False
#         self.rerun = True
#         self.graph_name = "email_assistant_hitl_memory_gmail"
#         self.url = "http://localhost:2024"  # or your LangGraph URL

# args = Args()
# await fetch_and_process_emails(args)
